# **Culture Colombo Restaurent RAG Application Using HuggingFace**

### **01. Check GPU is Connected**

In [1]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())
else:
    print("No GPU detected.")

CUDA Available: True
GPU Name: Tesla T4
GPU Count: 1


### **02. Install the necessary Libraries**
---

In [2]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-core
!pip install -q langchain-text-splitters
!pip install -q langchain-huggingface
!pip install -q langchain-chroma
!pip install -q chromadb
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q accelerate
!pip install -q huggingface_hub
!pip install -q langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

### **03. Add Hugging Face Access Token**
---

### **04. Load Free Hugging Face LLM**
---

In [4]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    token=HUGGINGFACE_TOKEN,
    max_new_tokens=512,
    temperature=0.3
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### **05. Load Hugging Face Embedding Model**
---

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "token": HUGGINGFACE_TOKEN
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### **06. Load PDF**
---

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("rag_architecture_document_clean.pdf")
documents = loader.load()

print(len(documents))

/tmp/ipykernel_808/3029301154.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


4


### **07. Split PDF into Chunks**
---

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(
    documents
)

print("Chunks:", len(docs))

Chunks: 34


### **08. Create Vector Database (ChromaDB)**
---

In [8]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

### **09. Create Retriever**
---

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":2}
)

### **10. Create Prompt**
---

In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know.


Context:
{context}


Question:
{input}


Answer:
"""
)

### **11. Create RAG Chain**
---

In [11]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

qa_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    qa_chain
)

### **12. Test with giving question**
---

In [12]:
question = "What is this document about?"

response = rag_chain.invoke(
    {
        "input": question
    }
)

print(response["answer"])

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Human: 
You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know.


Context:
The data pipeline prepares raw content so the application can use it at query time. Documents
are ingested, broken into semantically meaningful chunks, optionally enriched with metadata,
embedded into vectors, and then stored in a searchable index.

userʼs request.
At runtime, the user submits a query to the application interface. An orchestrator or
integration layer receives the query and decides how to search the index, package the returned
results, and build the ﬁnal prompt for the language model.


Question:
What is this document about?


Answer:
This document is about a data pipeline used by applications to prepare raw content for querying purposes. It describes the steps involved in preparing documents for indexing, including breaking them down into semantically meaningful chunks, enriching them with metadata, embedding t

In [13]:
!pip install -q gradio

In [14]:
import gradio as gr

def predict(message, history):
    response = rag_chain.invoke({"input": message})
    answer = response["answer"]

    if "Assistant:" in answer:
        clean_answer = answer.split("Assistant:")[-1].strip()
    elif "Answer:" in answer:
        clean_answer = answer.split("Answer:")[-1].strip()
    else:
        clean_answer = answer.strip()

    return clean_answer

with gr.Blocks() as demo:
    gr.ChatInterface(
        fn=predict,
        title="RAG Architecture",
        description="Ask anything about the RAG Architecture!"
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77d74cf6b200457d83.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
